# Supplementary Figure 10

PREDICT metabolite-enzyme pairs for 85 patients ordered by the metagenomic Enterotype Dysbiosis Score (EDS): three metabolite-enzyme (KO) pairs selected post hoc from a screen of the same cohort, with EDS, human-signal and antibiotic tracks. Values are standardized within feature and clipped at ±2. The v30 figure is a single panel; the former panel b (the 19-feature overview, identical to Fig. 4h) was removed.

Run from the repository root with `python paper.py build SFig10`. The antibiotic track and the covariate-adjusted statistics need the controlled input named in `controlled/CONTROLLED_INPUTS.md` (`FASTALAKE_CONTROLLED_DIR`); without it those steps print SKIPPED and every other number is still recomputed. Because the figure shows antibiotic state per patient, the notebook does not display it: the render is written to the gitignored `build/SFig10/`, and the v30 image it is checked against is read from `FASTALAKE_CONTROLLED_DIR`.

In [ ]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()

## Panel a

The single panel of the figure (printed without a letter): three post hoc metabolite-KO pairs (six heatmap rows) for 85 patients in EDS order, under three tracks: EDS, human-assigned share of LFQ intensity, and antibiotic state (controlled input).

Claim (v30 paragraph 73): "Across 85 patients, glucuronic acid, tryptophan and glycochenodeoxycholic acid were inversely associated with protein signal assigned to glucuronate isomerase, tryptophanase and bile salt hydrolase, respectively (Spearman ρ=−0.53, −0.44 and −0.35; 84 patients for tryptophan)."

n: 85 patients (84 for the tryptophan pair), one earliest matched observation per patient. Test: Spearman rank correlation on positive finite values, two-sided t approximation, Benjamini-Hochberg within the complete native KO pair family. The figure is written to `build/SFig10/panel_a.png` and `.pdf` and not displayed (controlled content).

In [ ]:
from IPython.display import display
d = panels.load(SD)
abx = panels.load_controlled(d["patients"])   # antibiotic state and clinical group, or None
display(panels.stat_a_unadjusted(d))
if abx is None:
    print("SKIPPED: the figure needs the antibiotic track (controlled input)")
else:
    res = panels.render(d, abx, OUT / "panel_a")
    print("510 heatmap cells identical to the producer's plotted values:", res["cells_identical"])
    print("plotted values identical to the v30 build table:", res["plotted_sha256"] == panels.V30_PLOTTED_SHA256)
    print("figure written to build/SFig10/panel_a.png and panel_a.pdf (not displayed: controlled content)")
print("clip at +-2, from the producer's plotted values:", panels.stat_clip(d))

## Statistics for panel a

Covariate sensitivity of the three pairs (v30 paragraphs 73 and 409): the 192 descriptive partial rank correlations (16 covariate combinations, two normalizations, pair-specific and common 84-patient subsets), the joint adjustment with its saved full-family q values, and the rule that produced the 21 candidate pairs. Estimates that involve antibiotic state or clinical group need the controlled input.

In [ ]:
from IPython.display import display
grid, n_common = panels.stat_a_adjustment_grid(d, abx)
saved = d["adjustments"]
print("estimates:", len(grid), "common subset:", n_common)
if abx is not None:
    print("max |rho - saved rho| over 192 estimates:", float((grid.rho - saved.rho).abs().max()))
main = grid[(grid.scope == "pair_complete") & (grid.normalization == "native")
            & grid.model.isin(["none", "depth", "antibiotics", "clinical", "host+depth+antibiotics+clinical"])]
display(main)
joint = panels.stat_a_joint_adjusted(d, abx)
display(joint)
fam, qdiff = panels.stat_family_q(d)
print("max |recomputed q - saved q| over all screen families:", qdiff)
display(fam)
print(panels.stat_family_sizes(d))
cand = panels.stat_candidates(d)
print("candidate pairs:", len(cand))
display(cand[["metabolite", "name", "protein_feature", "n", "rho", "q_augmented", "m_rho", "m_q", "k_rho", "k_q"]])

In [ ]:
import os, shutil
from tools.check_pixels import compare
ctrl = os.environ.get("FASTALAKE_CONTROLLED_DIR")
ctrl = Path(ctrl) if ctrl else None
if not fl._HAVE_STYLE:
    print("pixel checks SKIPPED: fonts absent")
elif abx is None or ctrl is None or not (ctrl / panels.EXPECTED_PNG).is_file():
    print("pixel checks SKIPPED: controlled input absent")
else:
    # expected images are controlled; the diff images stay in build/ (gitignored),
    # only the JSON receipts are kept in checks/
    (UNIT / "checks").mkdir(exist_ok=True)
    for new, exp, stem in [("panel_a.png", panels.EXPECTED_PNG, "SFig10_v33"),
                           ("panel_a.pdf", panels.EXPECTED_PDF, "SFig10_v33_pdf")]:
        r = compare(OUT / new, ctrl / exp, OUT / stem, dpi=300)
        shutil.copyfile(OUT / (stem + ".json"), UNIT / "checks" / (stem + ".json"))
        print(stem, r["verdict"], r.get("differing_pixels"))
print(panels.stat_counts(d))
for row, value, func in panels.numbers(d):
    print(row, repr(value), func)